Correlation Coefficient:
- https://www.scribbr.com/statistics/correlation-coefficient/

Categorical data processing:
- https://www.datacamp.com/tutorial/categorical-data




## Imports

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats.contingency as ssc
import plotly.express as px
from typing import Literal
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from dython.nominal import associations, cramers_v
from scipy import stats
import os

## Helper Functions

### Split Dataset for LOS and Mortality

In [ ]:
def split_data(df: pd.DataFrame):
    for e in ["Outcome", "DurationHospStayDay"]:
        if e not in df.columns.to_list():
            raise Exception("Feature: [{}] is required in dataset".format(e))
    
    mortality_df = df.copy(deep=True)
    mortality_df = mortality_df.drop(columns=["DurationHospStayDay"])
    
    los_df = df.copy(deep=True)
    los_df = los_df[los_df["Outcome"] == "Alive"]
    los_df = los_df.drop(columns=["Outcome"])
    
    return mortality_df, los_df

# Test code below
test_df = pd.DataFrame({
    "Name": ["Dave", "John", "Chris"],
    "DurationHospStayDay": [10, 20, 30],
    "Outcome": ["Alive", "Dead", "Alive"]
})

mortality_df, los_df = split_data(test_df)
display(mortality_df)
display(los_df)

### Include Labels in Feature List

In [ ]:
def include_labels(feature_list: list[str]):
    for e in ["Outcome", "DurationHospStayDay"]:
        if e not in feature_list:
            feature_list.append(e)
    
    return feature_list

# test code below
include_labels(["A"])

### Encoding

In [ ]:
def make_outcome_numerical(df: pd.DataFrame):
    df_copy = df.copy(deep=True)
    label_encoder = LabelEncoder() # alive = 0, dead = 1
    label_encoder.fit(df_copy["Outcome"])
    df_copy["Outcome"] = label_encoder.transform(df_copy["Outcome"])
    return df_copy

# test_df = make_outcome_numerical(sec_1_2_nominal_df)
# display(test_df.head(5)["Outcome"])
# display(sec_1_2_nominal_df.head(5)["Outcome"])

In [ ]:
def make_LOS_categorical(df: pd.DataFrame, classes: int = 10):
    if "DurationHospStayDay" not in df.columns.to_list():
        raise Exception("[DurationHospStayDay] not found in data frame")
    
    df_copy = df.copy(deep=True)
    df_copy["DurationHospStayDay"] = pd.qcut(df_copy["DurationHospStayDay"], q=classes, labels=range(classes))
    df_copy["DurationHospStayDay"] = df_copy["DurationHospStayDay"].astype("UInt8")
    return df_copy

# test code
test_df = pd.DataFrame({
    "Name": ["Dave", "John", "Chris"],
    "DurationHospStayDay": [1, 50, 100]
})
display(test_df)
test_df = make_LOS_categorical(test_df)
display(test_df)

In [ ]:
def one_hot_encode_data(df: pd.DataFrame, exclude_columns: list[str] = []):
    df = df.copy(deep=True)
    col_list = df.columns.copy(deep=True).to_list()
    
    for e in col_list:
        if e in exclude_columns:
            col_list.remove(e)
            
    return pd.get_dummies(df, columns=col_list)

# test code
test_df = pd.DataFrame({
    "Name": ["Dave", "John", "Chris"], 
    "Gender": ["M", "M", "M"]
})
test_df = one_hot_encode_data(test_df)
test_df

In [ ]:
def ordinal_encode_data(df: pd.DataFrame, categories_order: list[list[str]] = None):
    df = df.copy(deep=True)
    if categories_order is None:
        encoder = OrdinalEncoder()
    else:
        encoder = OrdinalEncoder(categories=categories_order)
    col_list = df.columns.copy(deep=True).to_list()
    
    if "DurationHospStayDay" in col_list:
        col_list.remove("DurationHospStayDay")
    
    df[col_list] = encoder.fit_transform(df[col_list])
    return df

# test code
test_df = pd.DataFrame({"Name": ["Dave", "John", "Chris"], "Gender": ["M","M","M"]})
test_df = ordinal_encode_data(test_df)
test_df

### Cramer's V

In [ ]:
def cramers_v_callable(arr1: pd.Series, arr2: pd.Series):
    ct = pd.crosstab(arr1, arr2)
    return ssc.association(ct, method="cramer")

### Point Biserial

In [ ]:
def point_biserial_correlation(df, target_column, title: str = None):
    # 计算与目标列的点双变量相关性
    correlation_results = {}
    
    for col in df.columns:
        print(col)
        if col != target_column and df[col].dtype != 'object':  # 排除目标列和非数值特征
            # 计算点双变量相关性
            corr, _ = stats.pointbiserialr(df[target_column], df[col])
            correlation_results[col] = corr
    
    # 将结果转换为 DataFrame
    corr_df = pd.DataFrame(list(correlation_results.items()), columns=["Feature", "Point Biserial Correlation"])

    # 按相关性值进行排序
    corr_df = corr_df.sort_values(by="Point Biserial Correlation", ascending=False)

    # 绘制条形图（Bar Chart）
    # fig = px.bar(corr_df, x="Feature", y="Point Biserial Correlation", 
    if title is None:
        title = f"Point Biserial Correlation with {target_column}"
    fig = px.bar(corr_df, x="Point Biserial Correlation", y="Feature", 
                 title=title,
                 labels={"Point Biserial Correlation": "Correlation Value", "Feature": "Features"},
                 color="Point Biserial Correlation", color_continuous_scale="RdBu")
    
    fig.update_layout(
        xaxis_title="Point Biserial Correlation",
        yaxis_title="Feature",
        showlegend=False,
        height=600
    )
    
    config = {
    'toImageButtonOptions': {
            'format': 'png', # one of png, svg, jpeg, webp
            'filename': 'custom_image',
            'height': 600,
            'width': 1000,
            'scale':6 # Multiply title/legend/axis/canvas sizes by this factor
        }
    }
    
    fig.show(config=config)

    return corr_df

### Compute Correlation

In [ ]:
def numerical_df_correlation_heatmap(df: pd.DataFrame, method: Literal['pearson', 'kendall', 'spearman']):
    corr_df = df.corr(method=method)
    fig = px.imshow(round(corr_df, 5), text_auto=True, aspect="auto")
    fig.show()
    return corr_df

In [ ]:
def numerical_df_correlation_bar(df: pd.DataFrame, method: Literal['pearson', 'kendall', 'spearman'], target_feature: str, title: str = None):
    corr_df = df.corr(method=method)
    corr_df = corr_df.drop(index=target_feature)

    correlation_column_name = f"{method.capitalize()} Correlation"
    
    corr_df = pd.DataFrame({
        correlation_column_name: corr_df[target_feature].to_list(),
        "Feature": corr_df.index
    })
    
    corr_df = corr_df.sort_values(by=correlation_column_name, ascending=False)
    
    if title is None:
        title = f"{correlation_column_name} with {target_feature}"
    
    fig = px.bar(corr_df, x=correlation_column_name, y="Feature", 
                 title=title,
                 labels={correlation_column_name: "Correlation Value", "Feature": "Features"},
                 color=correlation_column_name, color_continuous_scale="RdBu")
    
    fig.update_layout(
        xaxis_title=correlation_column_name,
        yaxis_title="Feature",
        showlegend=False,
        height=600
    )
    
    config = {
    'toImageButtonOptions': {
            'format': 'png', # one of png, svg, jpeg, webp
            'filename': 'custom_image',
            'height': 600,
            'width': 1000,
            'scale':6 # Multiply title/legend/axis/canvas sizes by this factor
        }
    }
    
    fig.show(config=config)
    
    return corr_df

In [ ]:
def categorical_df_correlation_heatmap(df: pd.DataFrame):
    corr_df = df.corr(method=cramers_v_callable)
    fig = px.imshow(round(corr_df, 5), text_auto=True, aspect="auto")
    fig.show()
    return corr_df

## Data Preparation

### Load Initial Data

In [ ]:
sec_123_df = pd.read_csv("../data/sample.csv")
column_nature_df = pd.read_csv("../data/column_nature.sample.csv")

is_nominal = (column_nature_df["columnNature"] == "nominal") | (column_nature_df["columnNature"] == "binary")

nominal_features_list = column_nature_df[is_nominal]["columnName"].to_list()
nominal_features_list = include_labels(nominal_features_list)

ordinal_features_list = column_nature_df[column_nature_df["columnNature"] == "ordinal"]["columnName"].to_list()
ordinal_features_list = include_labels(ordinal_features_list)

numerical_features_list = column_nature_df[column_nature_df["columnNature"] == "numerical"]["columnName"].to_list()
numerical_features_list = include_labels(numerical_features_list)

### Segment data based on nature to apply suitable correlation study

In [ ]:
sec_1_2_3_nominal_df = sec_123_df.filter(nominal_features_list)
sec_1_2_3_ordinal_df = sec_123_df.filter(ordinal_features_list)
sec_1_2_3_numerical_df = sec_123_df.filter(numerical_features_list)

## Numerical (Pearson)

### Outcome

In [ ]:
# dummy encode outcome
mortality_df, los_df = split_data(sec_123_df.filter(items=numerical_features_list))
mortality_df = pd.get_dummies(mortality_df, columns=["Outcome"])

In [ ]:
corr_df = numerical_df_correlation_heatmap(mortality_df, "pearson")

In [ ]:
corr_ranked = corr_df["Outcome_Alive"].sort_values()
display(corr_ranked.head(10))

corr_ranked = corr_df["Outcome_Alive"].sort_values(ascending=False)
display(corr_ranked.head(10))

In [ ]:
corr_ranked = corr_df["Outcome_Dead"].sort_values()
display(corr_ranked.head(10))

corr_ranked = corr_df["Outcome_Dead"].sort_values(ascending=False)
display(corr_ranked.head(10))

### Outcome (Point Biserial)

In [ ]:
mortality_df = mortality_df.drop(columns=["Outcome_Alive"])
correlation_df = point_biserial_correlation(mortality_df, "Outcome_Dead", "Point Biserial Correlation - Mortality")
correlation_df = correlation_df.sort_values(by="Point Biserial Correlation", ascending=False)

# Display the top 10 features with the highest correlation to the Outcome
display(correlation_df.head(10))
display(correlation_df.tail(10))

### LOS

Only check alive cases

In [ ]:
corr_df = numerical_df_correlation_heatmap(los_df, "pearson")

In [ ]:
corr_ranked = corr_df["DurationHospStayDay"].sort_values()
display(corr_ranked.head(10))

corr_ranked = corr_df["DurationHospStayDay"].sort_values(ascending=False)
display(corr_ranked.head(10))

In [ ]:
corr_ranked = pd.Series(np.absolute(corr_df["DurationHospStayDay"]))
corr_ranked = corr_ranked.sort_values(ascending=False)
display(corr_ranked.head(10))

In [ ]:
corr_df = numerical_df_correlation_bar(los_df, "pearson", "DurationHospStayDay", "Pearson Correlation - Length of Stay")

## Categorical - Nominal (Cramer's V)

In [ ]:
mortality_df, los_df = split_data(sec_123_df.filter(items=nominal_features_list))

### Outcome

In [ ]:
mortality_df = ordinal_encode_data(mortality_df)
corr_matrix = categorical_df_correlation_heatmap(mortality_df)

In [ ]:
corr_matrix["Outcome"].sort_values(ascending=False).head(20)

### LOS

In [ ]:
LOS_CLASSES = 5
los_df = make_LOS_categorical(los_df, LOS_CLASSES)
los_df = ordinal_encode_data(los_df)

In [ ]:
corr_matrix = categorical_df_correlation_heatmap(los_df)

In [ ]:
display(corr_matrix["DurationHospStayDay"].sort_values(ascending=False).head(5))

## Categorical - Ordinal (Spearman)

In [ ]:
mortality_df, los_df = split_data(sec_1_2_3_ordinal_df)
mortality_df = ordinal_encode_data(mortality_df, [["0", "< 1 hr", "1-2 hrs", "> 2 hrs"], ["Alive", "Dead"]])
mortality_df

### Outcome

In [ ]:
numerical_df_correlation_heatmap(mortality_df, method="spearman")

### LOS

In [ ]:
los_df = ordinal_encode_data(los_df, [["0", "< 1 hr", "1-2 hrs", "> 2 hrs"]])

In [ ]:
numerical_df_correlation_heatmap(los_df, method="spearman")

## Test library vs manual

conclusion, the library is included with correction https://www.sciencedirect.com/science/article/abs/pii/S1226319212001032

therefore the slight difference in value

### Manual

In [ ]:
test_df = pd.DataFrame({
    "Race": [0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2],
    "Obesity": [0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,1,1,1,1,1,1,1,1,0],
    "Country": [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1]
})

# Expectation: Obesity should have high association with Race and low association with Country

corr = test_df.corr(method=cramers_v_callable)
fig = px.imshow(round(corr, 5), text_auto=True, aspect="auto")
fig.show()

### Library

In [ ]:
associations(test_df,nominal_columns=["Race", "Obesity", "Country"])

In [ ]:
cramers_v(test_df["Race"], test_df["Obesity"])